In [0]:
pip install openpyxl

1 check data

In [0]:
import requests, pandas as pd
from pathlib import Path

URL = "https://www.scb.se/contentassets/c4b8142033a9440ca53725ca32321a74/kommungruppsindelning-2023.xlsx"
p = Path("/Volumes/laddstolpar_df/landing/raw/seeds/skr_kommungrupp/kommungruppsindelning-2023.xlsx")

if not (p.exists() and p.stat().st_size > 0):                 # static file: download once
    r = requests.get(URL, timeout=60)
    r.raise_for_status()
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(r.content)                                  # unchanged bytes
print(p, p.stat().st_size, "bytes")

xl = pd.ExcelFile(p)                                          # needs openpyxl ⚠️ (else: %pip install openpyxl)
print("Sheets:", xl.sheet_names)
for s in xl.sheet_names:
    print(f"\n--- {s}")
    print(pd.read_excel(p, sheet_name=s, header=None, nrows=8).to_string())

2 kommuner -> bronze

In [0]:
from pyspark.sql import functions as F

skr = pd.read_excel(p, sheet_name="Bilaga1 Lista alla kommuner", dtype=str)   # str keeps '0180'
assert list(skr.columns.str.strip()) == ["Gruppkod", "Kommunkod", "Kommunnamn", "Huvudgrupp", "Kommungrupp 2023"], skr.columns.str.strip()
skr.columns = ["gruppkod", "kommun_kod", "kommun_namn", "huvudgrupp", "kommungrupp"]
skr = skr.dropna(subset=["kommun_kod"])                     # in case of footnote rows

print("rows:", len(skr), "| code lengths:", skr["kommun_kod"].str.len().value_counts().to_dict())
print(skr.groupby(["gruppkod", "kommungrupp"]).size())

(spark.createDataFrame(skr)
    .withColumn("_file", F.lit(str(p)))
    .withColumn("_ingested_at", F.current_timestamp())
    .write.mode("overwrite").saveAsTable("laddstolpar_df.bronze.seed_skr_kommungrupp"))

3 Prize zone seeds

In [0]:
import shutil

SEED_SRC = Path.cwd() / "seeds"                     # the Git folder; the notebook sits in the repo root
print("Repo seeds:", SEED_SRC, [x.name for x in SEED_SRC.glob("*.csv")])
LAND = Path("/Volumes/laddstolpar_df/landing/raw/seeds/elomrade")
LAND.mkdir(parents=True, exist_ok=True)

for name in ["elomrade_lan_default.csv", "elomrade_kommun_override.csv"]:
    shutil.copyfile(SEED_SRC / name, LAND / name)   # Git is the master copy
    table = "laddstolpar_df.bronze.seed_" + name.removesuffix(".csv")
    (spark.createDataFrame(pd.read_csv(LAND / name, dtype=str, keep_default_na=False))
        .withColumn("_file", F.lit(str(LAND / name)))
        .withColumn("_ingested_at", F.current_timestamp())
        .write.mode("overwrite").saveAsTable(table))
    print(table, spark.table(table).count())        # expect 21 and 79

4 - checks

In [0]:
%sql
-- 1) Lookup codes exist and names match SKR
SELECT o.kommun_kod, o.kommun_namn, s.kommun_namn AS skr_namn
FROM laddstolpar_df.bronze.seed_elomrade_kommun_override o
LEFT JOIN laddstolpar_df.bronze.seed_skr_kommungrupp s USING (kommun_kod)
WHERE s.kommun_kod IS NULL OR s.kommun_namn <> o.kommun_namn;

-- 2) SKR and SCB agree on the 290 kommuner
WITH scb AS (SELECT DISTINCT region AS kommun_kod FROM laddstolpar_df.bronze.scb_tab628 WHERE length(region) = 4)
SELECT 'only SKR' AS side, kommun_kod FROM laddstolpar_df.bronze.seed_skr_kommungrupp
WHERE kommun_kod NOT IN (SELECT kommun_kod FROM scb)
UNION ALL
SELECT 'only SCB', kommun_kod FROM scb
WHERE kommun_kod NOT IN (SELECT kommun_kod FROM laddstolpar_df.bronze.seed_skr_kommungrupp);

-- 3) Every kommun gets a zone
SELECT COALESCE(o.elomrade_primary, l.elomrade_default) AS elomrade, COUNT(*) AS kommuner
FROM laddstolpar_df.bronze.seed_skr_kommungrupp k
LEFT JOIN laddstolpar_df.bronze.seed_elomrade_kommun_override o USING (kommun_kod)
LEFT JOIN laddstolpar_df.bronze.seed_elomrade_lan_default l ON l.lan_kod = substr(k.kommun_kod, 1, 2)
GROUP BY 1
ORDER BY 1;